<a href="https://colab.research.google.com/github/2k25cse2512541-wq/assignment1-flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k25cse2512541-wq/assignment1-flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

* **Lane Chosen:** Lane 2 — Refresh / Content Opportunity Scoring
* **Task Type:** Binary Classification + Score-Based Ranking (Supervised Learning)
* **Why:** The model predicts whether a page exhibits a declining traffic trend (`is_declining`), but operationally, human editors need an ordered queue. Machine learning scores candidates by predicted probability of decay weighted by search demand, generating an actionable top-K review list.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* **Target Label:** `is_declining_label`
* **Definition:** Derived directly from observed data: `1` if `trend_direction == 'down'` and `0` otherwise.
* **Proxy Nature:** In this initial starter slice, `trend_direction == 'down'` serves as our proxy label for content decay. In full production pipelines, this label evolves into a forward-looking target (predicting traffic drop in window $T+1$ based on window $T$ features) to avoid feature-target leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

* **Primary Metric:** `Precision@50` (and `Average Precision / AUPRC`)
* **Secondary Metric:** `ROC AUC`
* **What Means "Good":** An editorial team can realistically only review 20–50 pages a week. Overall accuracy is uninformative due to class distribution. `Precision@50` measures whether the top 50 candidates flagged by the model actually warrant a refresh, directly maximizing editorial time and yield.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

* **Unit of Analysis:** **1 row = 1 unique content item (URL / page)** for a specific client.
* **Granularity:** Each row represents a distinct URL's aggregate performance metrics (impressions, clicks, average position, content age) over a 90-day observation window.
* **Target Column Sketch:** We construct `is_declining_label` directly from `trend_direction` (`1` if `down`, `0` otherwise).

In [ ]:
import pandas as pd

# Load dataset directly from repository
url = "https://raw.githubusercontent.com/2k25cse2512541-wq/assignment1-flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Sketch binary target column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Filter for active pages (aligned with starter protocol)
active_df = df[df['impressions_90d'] > 0].copy()

# Print grain demonstration and shape
print(f"Dataset Grain: 1 row = 1 content item (page)")
print(f"Total active content items analyzed: {len(active_df):,}\n")

# Display key columns proving unit of analysis
sample_cols = ['content_id', 'client_id', 'impressions_90d', 'avg_position', 'content_age_days', 'trend_direction', 'is_declining_label']
active_df[sample_cols].head(5)

Dataset Grain: 1 row = 1 content item (page)
Total active content items analyzed: 30,000



,content_id,client_id,impressions_90d,avg_position,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,263,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

* **Rule Limitations:** A fixed heuristic (e.g., `IF content_age_days > 180 AND impressions_90d > 500`) relies on rigid, hardcoded cutoffs. It treats a page with 501 impressions identically to one with 50,000, and completely misses non-linear interactions across age, position drift, and click-through rates.
* **ML Advantage:** Supervised models (like Random Forests or Gradient Boosting) learn complex, non-linear decision boundaries across multiple continuous features (`avg_position`, `content_age_days`, `ctr`, and `trend_pct`) simultaneously without arbitrary hard thresholds.
* **Operational Impact:** Fixed rules flag thousands of false positives, wasting valuable editorial hours on healthy pages. ML models generate a probabilistic risk score, directly optimizing **Precision@50** to rank the top 50 pages most in need of attention.


In [ ]:
# Evaluate a typical naive fixed rule vs ground truth labels
rule_matches = active_df[(active_df['content_age_days'] > 180) & (active_df['impressions_90d'] > 500)]
rule_precision = rule_matches['is_declining_label'].mean()

print(f"Total pages flagged by naive fixed rule: {len(rule_matches):,}")
print(f"Naive Rule Precision across flagged set: {rule_precision:.3f}")
print(f"True declining pages in sample: {rule_matches['is_declining_label'].sum():,} / {len(rule_matches):,}")

Total pages flagged by naive fixed rule: 9,885
Naive Rule Precision across flagged set: 0.539
True declining pages in sample: 5,326 / 9,885


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.